In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [2]:
path = "/srv/models/phi3"

In [3]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

In [4]:
model = AutoModelForCausalLM.from_pretrained(
    path,
    quantization_config=bnb,
    device_map="cuda",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
tokenizer = AutoTokenizer.from_pretrained(path)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
from transformers import pipeline

In [7]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=5000,
    do_sample=False
)
messages = [
    {"role": "user", "content": "generate a program in C that analyzes the 5 given poker cards and saying whether its flash, straight, pair etc. cards should be provided via user input in stdin"}
]
output = generator(messages)

You are not running the flash-attention implementation, expect numerical differences.


In [8]:
print(output[0]["generated_text"])

 Certainatively! Below is a simple C program that reads 5 poker cards from the standard input (stdin), analyzes them, and then outputs whether the hand is a flush, straight, pair, etc. The program assumes that the cards are provided in a single line, separated by spaces, and in the format "RankSuit" (e.g., "10S" for the 10 of Spades).

```c
#include <stdio.h>
#include <string.h>
#include <stdlib.h>

#define MAX_CARDS 5
#define MAX_RANK 13
#define MAX_SUIT 4

typedef enum {
    FLUSH,
    STRAIGHT,
    PAIR,
    TWO_PAIR,
    THREE_OF_A_KIND,
    STRAIGHT_FLUSH,
    FULL_HOUSE,
    FOUR_OF_A_KIND,
    LONELY_PAIR
} HandType;

typedef struct {
    int rank;
    int suit;
} Card;

int compareRank(const void *a, const void *b) {
    Card *cardA = (Card *)a;
    Card *cardB = (Card *)b;
    return cardA->rank - cardB->rank;
}

int compareSuit(const void *a, const void *b) {
    Card *cardA = (Card *)a;
    Card *cardB = (Card *)b;
    return cardA->suit - cardB->suit;
}

int isSameSuit(Card